# Overview: Calculations Notebook

This notebook serves as a central workspace for all **quantitative calculations** in my research project on multi-agent financial forecasting system (MASFIN). Its purpose is to document, reproduce, and validate the numerical results that support each stage of the system’s design and evaluation.

## Importance of this File
- **Reproducibility**: All metrics and calculations are outline here and on our Github (EDIT) 
- **Transparency**: Each step is explicitly coded, making it clear how results are derived from raw data (e.g., Yahoo Finance).  
- **Integration**: The outputs generated here (returns, volatility, Sharpe ratio, drawdown, z-scores, etc.) feed directly into the MASFIN crews—particularly the **Analysis**, **Timing**, and **Portfolio** stages.  
- **Bias Control**: By centralizing and standardizing computations, this notebook helps reduce errors and prevents selective reporting of results.  
- **Reference**: It acts as a point of truth for validating results across weeks and comparing performance against benchmarks like the NASDAQ
In short, this file is not just a scratchpad for calculations but a **formal record of the quantitative calculations** of the research.

In [ ]:
!pip install yfinance

In [ ]:
!python -m pip install --upgrade pip

## 21-Day Stock Metrics Extraction (MASFIN Data Pipeline)

This script retrieves **21-day performance metrics** for a list of tickers and saves the results to an Excel file.  
It is part of the **Analysis Crew** in MASFIN, where quantitative indicators are standardized and passed to downstream agents (Beginning and End Snapshots).

### What the Code Does
1. **Imports Data**  
   Uses `yfinance` to download historical daily stock prices for the past ~30 calendar days (~21 trading days).  
   - `tickers`: list of stock symbols to analyze.  
   - `start_date`, `end_date`: define the data window.

2. **Computes Metrics** for each ticker:
   - **21-Day Return**: percentage change in closing prices over the window.  
   - **Volatility (Annualized)**: standard deviation of daily returns scaled to 252 trading days.  
   - **Sharpe Ratio**: annualized return per unit of volatility (risk-free rate assumed 0).  
   - **Max Drawdown**: largest observed decline from a peak to trough.  
   - **Momentum**: absolute price change over the period.

3. **Handles Missing Data**  
   If a ticker has insufficient history (<10 days), the script leaves its metrics blank.

4. **Outputs Results**  
   - Stores results in a Pandas DataFrame.  
   - Exports to Excel as:  
     ```
     Stock_Metrics_21Day_YYYY-MM-DD.xlsx
     ```
   - Includes columns: `Ticker, 21-Day Return, Volatility, Sharpe Ratio, Max Drawdown, Momentum`.

### How to Use in MASFIN
- **Input**: populate the `tickers` list with symbols passed from the **Screening Crew** or prior-week portfolio survivors.  
- **Run**: execute the script to generate standardized metrics.  
- **Output**: the Excel file is passed as structured input to the **Analysis Crew agents**, ensuring consistency across calculations.  
- **Integration**: this file provides the core quantitative metrics referenced in *Appendix A* of the MASFIN paper and is used in portfolio evaluation, risk adjustment, and timing decisions.

In [ ]:
import yfinance as yf
import pandas as pd
from datetime import date, timedelta

# Define tickers
tickers = []
# Date range for last 21 trading days (~30 calendar days)
end_date = date.today()
start_date = end_date - timedelta(days=30)
# Collect metrics
results = []
for ticker in tickers:
    try:
        df = yf.download(ticker, start=start_date, end=end_date)
        if len(df) < 10:
            results.append([ticker, "", "", "", "", ""])
            continue

        df['Return'] = df['Close'].pct_change()
        total_return = round((df['Close'].iloc[-1] / df['Close'].iloc[0]) - 1, 6)
        volatility = round(df['Return'].std() * (252 ** 0.5), 6)
        sharpe_ratio = round((df['Return'].mean() * 252) / (df['Return'].std() * (252 ** 0.5)), 6)
        max_drawdown = round(((df['Close'] / df['Close'].cummax()) - 1).min(), 6)
        momentum = round(df['Close'].iloc[-1] - df['Close'].iloc[0], 6)

        results.append([ticker, total_return, volatility, sharpe_ratio, max_drawdown, momentum])
    except:
        results.append([ticker, "", "", "", "", ""])

# Create DataFrame
columns = ["Ticker", "21-Day Return", "Volatility", "Sharpe Ratio", "Max Drawdown", "Momentum"]
df = pd.DataFrame(results, columns=columns)
# Export to Excel
today_str = date.today().isoformat()
df.to_excel(f"Stock_Metrics_21Day_{today_str}.xlsx", index=False)
df

## Beta and Alpha Estimation (MASFIN Data Pipeline)

This script calculates **market-based risk metrics** (Beta and Alpha) for a list of tickers using regression analysis against the **S&P 500 index**.  
It forms part of the **Analysis Crew** in MASFIN, providing inputs for portfolio evaluation and risk-adjusted performance measurement (Beginning and End Snapshots).

### What the Code Does
1. **Downloads Market Data**  
   - Uses `yfinance` to fetch the S&P 500 index (`^GSPC`) as a proxy for the overall market.  
   - Computes daily **Market Returns**.

2. **Processes Stock Data**  
   - For each ticker in the list, downloads daily prices.  
   - Computes daily **Stock Returns**.  
   - Aligns stock returns with market returns (merge on date index).

3. **Runs Regression**  
   - Fits an **Ordinary Least Squares (OLS)** model:  
     Stock Returnₜ = α + β × Market Returnₜ + εₜ 
   - Extracts:  
     - **Beta (β)**: sensitivity of the stock to market movements.  
     - **Alpha (α)**: stock-specific return unexplained by market movements.

4. **Handles Errors**  
   - If data is missing or merge fails, fills results with blanks.  
   - Prints log messages for debugging (e.g., `[NO DATA]` or `[ERROR]`).

5. **Outputs Results**  
   - Saves results to an Excel file:  
     ```
     Beta_Alpha_21Day_FIXED_CLEAN.xlsx
     ```  
   - Includes columns: `Ticker, Beta, Alpha`.

### How to Use in MASFIN
- **Input**: provide the `tickers` list from the **Screening Crew** and previous weeks portfolio.  
- **Run**: execute the script to compute each stock’s Beta and Alpha relative to the S&P 500.  
- **Output**: results are saved to Excel for structured use by the **Analysis Crew agents**.  
- **Integration**:  
  - **Beta** informs volatility and market-sensitivity analysis.  
  - **Alpha** helps identify idiosyncratic performance independent of market movement.  
  - Both are referenced in *Appendix A* and support the **Risk Adjuster** and **Portfolio Crew** in constructing a balanced portfolio.

In [ ]:
import statsmodels.api as sm

tickers = []
end_date = date.today()
start_date = end_date - timedelta(days=30)
# Download market (S&P 500)
market_df = yf.download("^GSPC", start=start_date, end=end_date)
market_df['Market Return'] = market_df['Close'].pct_change()
market_df = market_df[['Market Return']].dropna()
market_df.columns = ['Market Return']  # flatten column index
results = []
for ticker in tickers:
    try:
        stock_df = yf.download(ticker, start=start_date, end=end_date)
        stock_df['Stock Return'] = stock_df['Close'].pct_change()
        stock_df = stock_df[['Stock Return']].dropna()
        stock_df.columns = ['Stock Return']  # flatten to match merge expectations

        merged = pd.merge(stock_df, market_df, left_index=True, right_index=True)
        if merged.empty:
            print(f"[NO DATA] {ticker} - merge produced empty result")
            results.append([ticker, "", ""])
            continue

        print(f"[{ticker}] {len(merged)} data points")

        X = sm.add_constant(merged['Market Return'])
        y = merged['Stock Return']
        model = sm.OLS(y, X).fit()

        alpha = round(model.params['const'], 6)
        beta = round(model.params['Market Return'], 6)
        results.append([ticker, beta, alpha])

    except Exception as e:
        print(f"[ERROR] {ticker} - {e}")
        results.append([ticker, "", ""])
# Save results
df = pd.DataFrame(results, columns=["Ticker", "Beta", "Alpha"])
df.to_excel("Beta_Alpha_21Day_FIXED_CLEAN.xlsx", index=False)
df

## Short-Term Technical Metrics (5-Day Window)

This script calculates **short-term technical indicators** for each ticker in MASFIN, using a 5-day window to capture very recent trends (Beginning and End Snapshots). It supplements the 21-day metrics by emphasizing short-horizon momentum and anomalies.

### What the code does
1. **Data Retrieval**  
   - Downloads the last 10 days of daily price and volume data for all tickers.  
   - Builds aligned DataFrames for adjusted closing prices and trading volumes.  

2. **Return and Momentum**  
   - **5-Day Return**: Percentage change between today’s price and the price 5 trading days ago.  
   - **Momentum**: Raw price difference over the last 5 days.  

3. **Deviation from Moving Average**  
   - **Price vs. 5-Day MA**: Measures how far the current price deviates from its 5-day moving average, highlighting overbought/oversold conditions.  

4. **Statistical Signal**  
   - **Z-Score of Return**: Standardized measure of the most recent return relative to the last 5 days, showing unusual price moves.  

5. **Volume Analysis**  
   - **Volume Trend**: Slope of a linear regression fit over the last 5 days of volume (positive = increasing activity).  
   - **Residual Volume**: Difference between observed and expected volume from that regression, capturing short-term shocks.  

6. **Final Output**  
   - All metrics are combined into a single `DataFrame` and rounded for readability.  
   - Results can be printed, stored, or passed forward into later MASFIN crews.  

### Role in MASFIN
- These short-term indicators are used by the **Analysis Crew** to confirm or reject signals from longer-horizon metrics.  
- They help detect **momentum bursts, abnormal volatility, or unusual trading activity** that might not show up in 21-day metrics.  
- By combining return, volume, and deviation indicators, the system can better filter out **noise vs. genuine signals** in short-term price action. 

In [ ]:
import numpy as np
from scipy.stats import zscore

# Tickers
tickers = []
# Download last 10 days of 1-day interval data
data = yf.download(tickers, period="10d", interval="1d", group_by='ticker', auto_adjust=True)

# Build price and volume DataFrames
adj_close = pd.DataFrame({ticker: data[ticker]['Close'] for ticker in tickers if ticker in data and 'Close' in data[ticker]})
volume = pd.DataFrame({ticker: data[ticker]['Volume'] for ticker in tickers if ticker in data and 'Volume' in data[ticker]})

# Drop tickers with missing prices
adj_close = adj_close.dropna(axis=1)
volume = volume[adj_close.columns]  # align volume with price, for Finacial Analysis with AI project

# 5-day return and momentum
return_5d = adj_close.iloc[-1] / adj_close.iloc[-6] - 1 if len(adj_close) >= 6 else pd.Series(index=adj_close.columns)
momentum = adj_close.iloc[-1] - adj_close.iloc[-6] if len(adj_close) >= 6 else pd.Series(index=adj_close.columns)

# 5-day moving average ratio
ma_5 = adj_close.rolling(window=5).mean()
price_vs_ma = adj_close.iloc[-1] / ma_5.iloc[-1] - 1 if len(ma_5) >= 5 else pd.Series(index=adj_close.columns)

# Z-score of most recent return
returns = adj_close.pct_change().dropna()
z_scores = returns.tail(5).apply(zscore).iloc[-1] if len(returns) >= 5 else pd.Series(index=adj_close.columns)

# Volume trend (slope from linear regression) and residuals
volume_trend = {}
residuals = {}
if len(volume) >= 5:
    for ticker in adj_close.columns:
        y = volume[ticker].tail(5)
        X = sm.add_constant(np.arange(len(y)))
        model = sm.OLS(y, X).fit()
        volume_trend[ticker] = model.params[1]
        residuals[ticker] = model.resid.iloc[-1]
else:
    volume_trend = {ticker: np.nan for ticker in adj_close.columns}
    residuals = {ticker: np.nan for ticker in adj_close.columns}

# Final DataFrame
metrics = pd.DataFrame({
    '5D Return': return_5d,
    'Z-Score (Return)': z_scores,
    'Volume Trend': pd.Series(volume_trend),
    'Residual (Volume)': pd.Series(residuals),
    'Price vs MA(5)': price_vs_ma
}).round(4)

print(metrics)

## Portfolio Percentage Change Between Dates

This script measures **percentage price changes** for a list of tickers over a defined date range (End Snapshot).  
It is useful for quickly checking portfolio performance across arbitrary periods, beyond the standardized 5-day or 21-day windows.

### What the code does
1. **Inputs**  
   - Accepts a list of tickers along with a custom `start_date` and `end_date`.  
   - Uses Yahoo Finance (`yfinance`) to fetch price data.  

2. **Computation per Ticker**  
   - Retrieves closing prices for the first and last trading days in the range.  
   - Calculates percent change:  
     Percent Change = ((P_end – P_start) / P_start) × 100  
   - Prints results for each ticker in the format:  
     ```
     TICKER: XX.XX% change | Start: $P_start → End: $P_end
     ```

3. **Portfolio-Level Summary**  
   - Computes the average percent change across all tickers analyzed.  
   - Displays a concise summary at the end.  

### Role in MASFIN
- Acts as a **sanity-check tool** for validating weekly or multi-week results against MASFIN’s more complex metric framework.  
- Can be applied to:  
  - **Backtests**, where the researcher wants to see raw returns across a chosen time window.  
  - **Performance attribution**, comparing MASFIN’s recommended portfolio vs. a benchmark index (e.g., S&P 500).  
- Unlike the 5-day and 21-day metric scripts, this function is flexible — it allows custom ranges, making it especially useful for **event-driven analysis** (e.g., before and after earnings announcements).

In [ ]:
import warnings

def stock_changes_between_dates(tickers, start_date, end_date):
    warnings.simplefilter(action='ignore', category=FutureWarning)

    changes = []

    for ticker in tickers:
        data = yf.download(ticker, start=start_date, end=end_date, progress=False)

        if data.empty or len(data) < 2:
            print(f"{ticker}: No sufficient data between {start_date} and {end_date}")
            continue

        try:
            start_price = data['Close'].iloc[0].item() if hasattr(data['Close'].iloc[0], 'item') else data['Close'].iloc[0]
            end_price = data['Close'].iloc[-1].item() if hasattr(data['Close'].iloc[-1], 'item') else data['Close'].iloc[-1]
            percent_change = ((end_price - start_price) / start_price) * 100
            changes.append(percent_change)

            print(f"{ticker}: {percent_change:.2f}% change | Start: ${start_price:.2f} → End: ${end_price:.2f}")
        except Exception as e:
            print(f"{ticker}: Error - {e}")

    if changes:
        avg_change = sum(changes) / len(changes)
        print(f"\nAverage percent change across portfolio: {avg_change:.2f}%")

tickers = []

start = '2025-XX-XX'
end = '2025-YY-YY'

stock_changes_between_dates(tickers, start, end)

## Benchmark Growth Comparison (Index Baselines)

This function is used in MASFIN to compare the performance of the system’s simulated portfolio against major U.S. indices (S&P 500, NASDAQ 100, and Dow Jones Industrial Average).  
The comparison begins on **June 16, 2025**, the official start date of this research simulation.

### Code Overview
- **`get_closest_trading_date`**  
  Ensures that any chosen date aligns with an actual market trading day by returning the next available trading date.  
  This accounts for weekends and holidays.

- **`compare_growth_from_june16`**  
  1. Downloads historical price data for each index using `yfinance`.  
  2. Identifies three anchor points:  
     - June 16, 2025 (simulation start date)  
     - Start date of the week under analysis  
     - End date of the week under analysis  
  3. Calculates how much a fixed investment (e.g., \$100) made on June 16 would have grown in value at the start and end of the week.  
  4. Reports percentage and dollar changes in index values over the week.

### Why It Matters in MASFIN
This function creates a **baseline benchmark** against which MASFIN’s weekly portfolio results can be measured.  
By simulating an initial investment in indices and tracking their growth, MASFIN can compare whether its portfolios demonstrate:  
- **Outperformance** (higher weekly returns than benchmarks)  
- **Underperformance** (lower returns than benchmarks)  

This ensures that MASFIN’s results are not evaluated in isolation but are contextualized against standard market movements, improving the rigor of the analysis.

In [ ]:
def get_closest_trading_date(data, target_date):
    """Return the next trading day on or after target_date."""
    target = pd.to_datetime(target_date)
    valid_dates = data.index[data.index >= target]
    return valid_dates[0] if not valid_dates.empty else None

def compare_growth_from_june16(start_date, end_date, initial_investment_date, investment_amount): #June 16 is the start date of this research simulation
    indices = {
        "S&P 500": "^GSPC",
        "NASDAQ 100": "^NDX",
        "Dow Jones": "^DJI"
    }

    start_dt = pd.to_datetime(start_date)
    end_dt = pd.to_datetime(end_date)
    invest_dt = pd.to_datetime(initial_investment_date)

    for name, symbol in indices.items():
        data = yf.download(symbol, start=invest_dt - pd.Timedelta(days=3), 
                           end=end_dt + pd.Timedelta(days=7), progress=False)

        if data.empty:
            print(f"{name}: No data downloaded at all.\n")
            continue

        june16_actual = get_closest_trading_date(data, invest_dt)
        start_actual = get_closest_trading_date(data, start_dt)
        end_actual = get_closest_trading_date(data, end_dt)

        if not june16_actual or not start_actual or not end_actual:
            print(f"{name}: Missing valid trading data for June 16, {start_date}, or {end_date}.\n")
            continue

        # Get scalar float values safely
        june16_price = data.loc[june16_actual]["Open"]
        start_price = data.loc[start_actual]["Open"]
        end_price = data.loc[end_actual]["Close"]

        june16_price = june16_price.item() if hasattr(june16_price, "item") else june16_price
        start_price = start_price.item() if hasattr(start_price, "item") else start_price
        end_price = end_price.item() if hasattr(end_price, "item") else end_price

        shares = investment_amount / june16_price
        value_start = shares * start_price
        value_end = shares * end_price
        change_dollars = value_end - value_start
        change_percent = (change_dollars / value_start) * 100

        print(f"{name}:")
        print(f"  Initial price on {june16_actual.date()}: ${june16_price:.2f}")
        print(f"  Value on {start_actual.date()}: ${value_start:.2f}")
        print(f"  Value on {end_actual.date()}: ${value_end:.2f}")
        print(f"  Change ({start_actual.date()} to {end_actual.date()}): {change_percent:.2f}% | ${change_dollars:.2f}\n")

# Example usage:
compare_growth_from_june16(
    start_date="2025-XX-XX",
    end_date="2025-YY-08",
    initial_investment_date="2025-ZZ-ZZ",
    investment_amount=100.00 #Example
)

## Benchmark Volatility (Standard Deviation of Daily Returns)

This code computes the **volatility** of major U.S. indices (S&P 500, NASDAQ 100, and Dow Jones Industrial Average) by calculating the **standard deviation of daily returns** within a given time window.

### Code Overview
1. **Index Dictionary**  
   Maps index names to their Yahoo Finance ticker symbols:  
   - S&P 500 → `^GSPC`  
   - NASDAQ 100 → `^NDX`  
   - Dow Jones → `^DJI`

2. **Download Data**  
   Historical price data is fetched for each index between a defined start and end date.

3. **Daily Returns**  
   Uses the percentage change of daily closing prices:  
   Rt = (Pt - Pt-1) / Pt-1

   
   where:
    - Rt = daily return on day t
    - Pt = price at time t
    - Pt-1 = price at previous time (t-1)

5. **Volatility (Standard Deviation)**  
   The dispersion of returns is measured as:  
   σ = sqrt( (1 / (N - 1)) * Σ (Rt - R̄)² )

   
   where:
    - Rt = daily return on day t
    - R̄ = average daily return over the time window
    - N = number of trading days in the window
### Why It Matters in MASFIN
Volatility provides a **risk baseline** for evaluating MASFIN’s weekly portfolios.  
- A higher standard deviation indicates larger fluctuations and greater uncertainty.  
- By comparing MASFIN’s portfolio risk against these benchmark volatilities, researchers can determine if MASFIN achieves **risk-adjusted outperformance** rather than only raw returns.  

In [ ]:
# Index symbols
indices = {
    "S&P 500": "^GSPC",
    "NASDAQ 100": "^NDX",
    "Dow Jones": "^DJI"
}

start_date = "2025-XX-XX"
end_date = "2025-YY-XX"

print("Standard Deviation of Daily Returns:")

for name, symbol in indices.items():
    try:
        data = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if data.empty or "Close" not in data:
            print(f"{name}: No data available")
            continue

        # Extract Close prices and compute returns
        close = data["Close"]
        returns = close.pct_change().dropna()

        # Compute and print standard deviation
        std_value = returns.std()
        print(f"{name}: {round(std_value, 4)}")

    except Exception as e:
        print(f"{name}: Error - {str(e)}")



## Portfolio Standard Deviation of Daily Returns

This code calculates the standard deviation of daily returns for a portfolio with predefined allocations.  

**Steps:**
1. **Define allocations**: The portfolio is represented as a dictionary of tickers with their percentage weights.  
2. **Download price data**: Using `yfinance`, historical adjusted closing prices (or closing prices if adjusted not available) are downloaded for all tickers over the given date range.  
3. **Clean data**: Any tickers without complete data (due to missing values or market holidays) are removed. The weights are re-normalized so the remaining tickers sum to 100%.  
4. **Compute returns**: Daily percentage returns are calculated for each stock.  
5. **Aggregate into portfolio return**: A weighted sum of daily returns is computed based on portfolio allocations.  
6. **Measure volatility**: The standard deviation of these portfolio returns is calculated, representing short-term portfolio risk.  

**Use in MASFIN:**  
This script allows the MASFIN system to measure weekly portfolio volatility, ensuring comparisons to indices or alternative portfolios are risk-adjusted. By feeding in the allocations produced by the Portfolio Crew, this code provides a quantitative check on how stable or volatile the constructed portfolio is over a given period.

In [ ]:
# Portfolio allocations (in %)

# Example Usage
allocations = {
    "AAPL": 10.53,
    "NVDA": 10.53,
    "GOOGL": 8.42,
    "MSFT": 8.42,
    "AMZN": 7.37,
    "AVGO": 7.37,
    "PFE": 7.37,
    "TMO": 6.32,
    "WMT": 6.32,
    "UNH": 5.26,
    "NFLX": 5.26,
    "RMD": 4.21,
    "DHR": 4.21,
    "CSCO": 3.16,
    "JNJ": 2.11,
    "KO": 2.11,
    "NKE": 1.03
}


weights = np.array(list(allocations.values())) / 100
tickers = list(allocations.keys())

start_date = "2025-08-04"
end_date = "2025-08-08"

# Download data
data = yf.download(tickers, start=start_date, end=end_date, progress=False)

# Fallback: if "Adj Close" doesn't exist, try "Close"
if "Adj Close" in data.columns:
    prices = data["Adj Close"]
elif "Close" in data.columns:
    prices = data["Close"]
else:
    raise ValueError("No adjusted or close price data found.")

# Drop columns with missing data (e.g., holidays or delisted tickers)
prices = prices.dropna(axis=1)
valid_tickers = prices.columns.tolist()

# Filter weights to only those tickers we actually got data for
weights_filtered = np.array([allocations[t] for t in valid_tickers]) / 100
weights_filtered = weights_filtered / weights_filtered.sum()  # re-normalize

# Calculate daily returns
returns = prices.pct_change().dropna()

# Portfolio return = weighted sum of individual returns
portfolio_returns = returns.dot(weights_filtered)

# Portfolio volatility (standard deviation)
std_dev = portfolio_returns.std()

print(f"Portfolio Std Dev of Daily Returns ({start_date} to {end_date}): {std_dev:.6f}")

# Weekly Profit Calculation (Manual Method)

To evaluate MASFIN’s weekly performance, we manually compute profit (or loss) for each stock holding based on whether the position was **purchased during the evaluation week** or **carried over from a prior week**. This distinction ensures consistency in how gains are attributed.

## Calculation Rules

1. **New Purchases (acquired during the week)**  
   Profit is calculated relative to the **average cost basis** at purchase:

   $$
   \text{Profit} = (\text{Close}_{\text{end of week}} - \text{Avg. Cost Basis}) \times \text{Shares}
   $$

2. **Existing Holdings (purchased before the week and held through)**  
   Profit is measured relative to the **opening price** of the first trading day in the week:

   $$
   \text{Profit} = (\text{Close}_{\text{end of week}} - \text{Open}_{\text{start of week}}) \times \text{Shares}
   $$

## Rationale
- **Fair Comparison Across Weeks**: This method aligns each holding with the correct entry point for that week, avoiding distortion from prior gains or losses.  
- **Inclusion of All Holdings**: Every stock that is active during the week (whether newly acquired or carried over) contributes to weekly profit.  
- **Benchmarking**: Results can be compared directly against indices (NASDAQ, S&P 500, Dow Jones) to track whether MASFIN outperforms the market on a week-by-week basis.  

This approach makes weekly profits **additive and consistent**, supporting reliable performance evaluation of MASFIN portfolios.